In [ ]:
import os
import pandas as pd
from pysus import SINAN

ANOS = [2023, 2024] # Exemplo
caminho_base = './data/raw/sinan'

# Colunas essenciais para incidência (economiza RAM)
COLUNAS_DESEJADAS = [
    'DT_NOTIFIC', 
    'DT_SIN_PRI', 
    'ID_MN_RESI', 
    'CLASSI_FIN', 
    'NU_ANO',
    'SG_UF_NOT'
]

def baixar_e_processar_dengue():
    sinan = SINAN().load()
    
    for ano in ANOS:
        print(f'--- Processando ano {ano} ---')
        try:
            # 1. Busca os arquivos no servidor
            files = sinan.get_files(dis_code='DENG', year=ano)
            
            if not files:
                print(f'Nenhum arquivo encontrado para {ano}.')
                continue
            
            # 2. Baixa e converte para Parquet (Salva no disco, não na RAM ainda)
            # O download retorna uma lista de caminhos de arquivos parquet salvos
            caminho_dir = f"{caminho_base}/dengue"
            os.makedirs(caminho_dir, exist_ok=True)
            
            print(f"Baixando arquivos para: {caminho_dir}")
            lista_arquivos_parquet = sinan.download(files, local_dir=caminho_dir)
            
            # 3. Lê apenas as colunas necessárias
            for parquet_path in lista_arquivos_parquet:
                print(f"Lendo arquivo otimizado: {os.path.basename(parquet_path)}")
                
                # O segredo está aqui: columns=COLUNAS_DESEJADAS
                df = pd.read_parquet(parquet_path, columns=COLUNAS_DESEJADAS)
                
                # Exemplo de pré-filtro para limpar memória (remove descartados - código 5)
                # O PDF diz que CLASSI_FIN = 5 é Descartado.
                df = df[df['CLASSI_FIN'] != '5'] 
                
                print(f"Dados carregados na memória: {df.shape[0]} linhas. Colunas: {list(df.columns)}")
                
                # Aqui você salvaria seu CSV processado ou continuaria a análise
                # df.to_csv(f'dengue_resumo_{ano}.csv', index=False)
                
        except Exception as e:
            print(f'Erro no ano {ano}: {e}')

if __name__ == "__main__":
    baixar_e_processar_dengue()
    print("Concluído.")

Buscando dados do SINAN (Dengue) - 2024
Arquivos encontrados para 2024: [RAIVBR24.dbc]










/home/gabrielgraciano/case-IA-grupo-02/projeto_grupo_02/.venv/lib/python3.12/site-packages/tqdm/std.py:533: TqdmWarning: clamping frac to range [0, 1]
  full_bar = Bar(frac,
RAIVBR24.parquet: 100%|██████████| 35.5/35.5 [00:00<00:00, 1.36kB/s]

Processo concluído.
